# Colab runner: one click per experiment

Open this notebook in Colab and choose **Runtime > Run all**. Never edit a cell: the job to run is read from the `colab-queue` branch
(`job.json`), the result bundle is pushed to the `colab-results` branch and copied to Drive (`MyDrive/crypto_runs/<job_id>/`).

One-time setup (token file on Drive), the `job.json` schema and how results are read: `docs/research/plan/COLAB_RUNNER.md`.
All logic lives in `tools/colab_runner.py`; this notebook only mounts Drive and calls it. The next code cell holds the only constants.

In [ ]:
REPO = "yuosef772424/crypto-signal-prediction"
TOKEN_PATH = "/content/drive/MyDrive/secrets/github_token.txt"
QUEUE_BRANCH = "colab-queue"
RESULTS_BRANCH = "colab-results"
WORKDIR = "/content/csp_runner"

## 1. Mount Drive, read the token, fetch the job and the runner code
The token goes into an HTTP header only (never a URL, never printed). The runner module is downloaded from the exact commit the job pins.

In [ ]:
import json, os, re, sys, urllib.parse, urllib.request
from google.colab import drive

drive.mount("/content/drive")
try:
    _tok = open(TOKEN_PATH, encoding="utf-8").read().strip()
except OSError:
    raise RuntimeError(f"token file not readable at {TOKEN_PATH}: see docs/research/plan/COLAB_RUNNER.md") from None

def _gh(path, ref=None, accept="application/vnd.github.raw"):
    url = f"https://api.github.com/repos/{REPO}/{path}" + (f"?ref={urllib.parse.quote(ref, safe='')}" if ref else "")
    req = urllib.request.Request(url, headers={"Authorization": "Bearer " + _tok, "Accept": accept,
                                               "X-GitHub-Api-Version": "2022-11-28", "User-Agent": "colab-runner"})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            return r.read().decode("utf-8")
    except Exception as e:
        raise RuntimeError(f"GitHub request for {path} failed: {type(e).__name__} {getattr(e, 'code', '')} "
                           "(token needs Contents read/write on this repo)") from None

_job_text = _gh("contents/job.json", QUEUE_BRANCH)
_queue_sha = _gh("commits/" + urllib.parse.quote(QUEUE_BRANCH, safe=""), accept="application/vnd.github.sha").strip()
try:
    _ref = str(json.loads(_job_text)["repo_ref"]).lower()
except Exception:
    raise RuntimeError("job.json is not valid JSON or has no repo_ref") from None
if not re.fullmatch(r"[0-9a-f]{40}", _ref):
    raise RuntimeError("job.json repo_ref must be a full 40-hex commit SHA")
os.makedirs(f"{WORKDIR}/bootstrap", exist_ok=True)
with open(f"{WORKDIR}/bootstrap/colab_runner.py", "w", encoding="utf-8") as fh:
    fh.write(_gh("contents/tools/colab_runner.py", _ref))
sys.path.insert(0, f"{WORKDIR}/bootstrap")
sys.modules.pop("colab_runner", None)
import colab_runner as R
del _tok, _gh
print("queue commit:", _queue_sha[:12], "| runner from:", _ref[:12])

## 2. Validate the job, refuse finished jobs, check out the pinned commit, install
A job id that already has `runs/<job_id>/` on `colab-results` is refused unless `job.json` sets `"rerun": true`.

In [ ]:
runner = R.Runner(R.github_url(REPO), R.read_token(TOKEN_PATH), WORKDIR, QUEUE_BRANCH, RESULTS_BRANCH)
runner.load(_job_text, _queue_sha)
runner.check_rerun()
runner.checkout()
runner.install()

## 3. Run the entry script (wall-clock guard, log streamed to a file)

In [ ]:
runner.execute()

## 4. Validate outputs, push to `colab-results`, copy to Drive, print the status
A failed run still pushes its log with status FAIL. The last block printed here is all that needs to be copied if anything is.

In [ ]:
result = runner.publish()